In [48]:
# Cell 1: Install Dependencies
!pip install -q streamlit langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu pypdf groq pandas python-dotenv flashrank

In [47]:
# Cell 2: Create Folder Structure
import os
os.makedirs('self_healing_rag/data', exist_ok=True)
os.makedirs('self_healing_rag/vectorstore', exist_ok=True)
os.makedirs('self_healing_rag/logs', exist_ok=True)

In [46]:
%%writefile self_healing_rag/document_loader.py
import os
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_and_split_docs(file_path, chunk_size=500, chunk_overlap=50):
    if file_path.endswith('.pdf'):
        loader = PyPDFLoader(file_path)
    else:
        loader = TextLoader(file_path)
    docs = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return text_splitter.split_documents(docs)

Overwriting self_healing_rag/document_loader.py


In [60]:
%%writefile self_healing_rag/vector_store.py
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import faiss

def create_vector_store(chunks, path="self_healing_rag/vectorstore"):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    # Use distance_strategy=MAX_INNER_PRODUCT with normalized embeddings for Cosine Similarity
    vector_db = FAISS.from_documents(chunks, embeddings, distance_strategy="COSINE")
    vector_db.save_local(path)
    return vector_db

Overwriting self_healing_rag/vector_store.py


In [66]:
%%writefile self_healing_rag/healing.py
class SelfHealer:
    def rewrite_query(self, client, query):
        # Strictly internal rewrite for retrieval optimization only
        prompt = f"Rewrite this search query to be more descriptive for vector retrieval: {query}. Output ONLY the rewritten query."
        completion = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )
        return completion.choices[0].message.content.strip()

    def generate_suggestions(self, client, query, context):
        # Generates user-friendly alternative questions
        prompt = f"""Original Question: {query}
Context Snippet: {context[:500]}

Based on the question and context, generate 3 better, more specific versions of the question that would help a RAG system find a more accurate answer.
Output ONLY the 3 questions as a bulleted list starting with '-'. No preamble."""
        completion = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )
        return completion.choices[0].message.content.strip().split('\n')

    def check_grounding(self, client, context, answer):
        prompt = f"Context: {context}\n\nAnswer: {answer}\n\nDoes the context support this answer? Reply only YES or NO."
        completion = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )
        return "YES" in completion.choices[0].message.content.upper()

Overwriting self_healing_rag/healing.py


In [51]:
%%writefile self_healing_rag/logger.py
import pandas as pd
import datetime
import os

def log_query(query, answer, score, actions, log_path="self_healing_rag/logs/query_logs.csv"):
    log_data = {
        "timestamp": [datetime.datetime.now()],
        "query": [query],
        "answer": [answer],
        "score": [score],
        "actions": [", ".join(actions)]
    }
    df = pd.DataFrame(log_data)
    df.to_csv(log_path, mode='a', index=False, header=not os.path.exists(log_path))

Overwriting self_healing_rag/logger.py


In [72]:
%%writefile self_healing_rag/rag_pipeline.py
import os
import numpy as np
from groq import Groq
from healing import SelfHealer
from logger import log_query
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

def normalize_score(score):
    # FAISS distance to similarity conversion
    # For COSINE, higher is better. We clip and ensure 0-1 range.
    return float(np.clip(score, 0, 1))

def run_rag_pipeline(user_query, api_key, threshold=0.7):
    client = Groq(api_key=api_key)
    healer = SelfHealer()
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_db = FAISS.load_local("self_healing_rag/vectorstore", embeddings, allow_dangerous_deserialization=True)

    actions = []
    suggestions = []

    # 1. Initial Retrieval
    docs_scores = vector_db.similarity_search_with_relevance_scores(user_query, k=5)
    top_score = normalize_score(docs_scores[0][1]) if docs_scores else 0

    # 2. Self-Healing: Query Rewriting & Suggestions
    search_query = user_query
    if top_score < threshold:
        actions.append(f"Low confidence ({top_score*100:.1f}%) detected")
        search_query = healer.rewrite_query(client, user_query)
        docs_scores = vector_db.similarity_search_with_relevance_scores(search_query, k=5)
        top_score = normalize_score(docs_scores[0][1]) if docs_scores else 0
        actions.append("Internal query rewrite performed")

        context_for_sugg = "\n".join([d[0].page_content for d in docs_scores[:2]])
        suggestions = healer.generate_suggestions(client, user_query, context_for_sugg)
        actions.append("Suggested better questions generated")

    context = "\n".join([d[0].page_content for d in docs_scores])

    gen_prompt = f"""Answer ONLY using context. QUESTION: {user_query} \nCONTEXT: {context}"""
    res = client.chat.completions.create(model="llama-3.1-8b-instant", messages=[{"role": "user", "content": gen_prompt}])
    answer = res.choices[0].message.content

    if not healer.check_grounding(client, context, answer):
        actions.append("Hallucination check failed - regenerating")
        res = client.chat.completions.create(model="llama-3.1-8b-instant", messages=[{"role": "user", "content": "STRICT: " + gen_prompt}])
        answer = res.choices[0].message.content

    log_query(user_query, answer, top_score, actions)
    return answer, docs_scores, actions, top_score, suggestions

Overwriting self_healing_rag/rag_pipeline.py


In [68]:
%%writefile self_healing_rag/app.py
import streamlit as st
import os
from document_loader import load_and_split_docs
from vector_store import create_vector_store
from rag_pipeline import run_rag_pipeline

st.set_page_config(page_title="Self-Healing RAG", layout="wide")
st.title("🛡️ Self-Healing RAG Pipeline")

with st.sidebar:
    st.header("Settings")
    key = st.text_input("Groq API Key", type="password")
    threshold = st.slider("Threshold (Suggestion Trigger)", 0.0, 1.0, 0.7)

up = st.file_uploader("Upload Resume/Document", type=['pdf', 'txt'])
if up and key:
    path = os.path.join("self_healing_rag/data", up.name)
    with open(path, "wb") as f: f.write(up.getbuffer())
    if st.button("Process Document"):
        with st.spinner("Indexing..."):
            create_vector_store(load_and_split_docs(path))
            st.success("Vector Store Ready!")

q = st.text_input("Ask a question:")
if q and key:
    with st.spinner("Running Pipeline..."):
        ans, sources, acts, score, suggestions = run_rag_pipeline(q, key, threshold)

        st.markdown(f"### Answer\n{ans}")

        col1, col2 = st.columns(2)
        with col1:
            st.metric("Confidence Score", f"{max(0, score)*100:.1f}%")
        with col2:
            st.write("**Healing Actions:**")
            for a in acts: st.write(f"✓ {a}")

        if suggestions:
            st.markdown("### 💡 Suggested Better Questions")
            for sug in suggestions:
                st.markdown(sug)

        with st.expander("Sources & Citations"):
            for d, s in sources:
                st.write(f"**Similarity Score: {max(0, s):.2f}**")
                st.info(d.page_content[:300] + "...")

Overwriting self_healing_rag/app.py


In [74]:
import os
import time
print("Restarting server to apply score normalization fixes...")
os.system('pkill streamlit')
os.system('pkill bore')
os.system('streamlit run self_healing_rag/app.py --server.port 8501 --server.address 0.0.0.0 > streamlit_log.txt 2>&1 &')
time.sleep(5)
!bore local 8501 --to bore.pub

Restarting server to apply score normalization fixes...
2026-06-11T07:17:48.423855Z  INFO bore_cli::client: connected to server remote_port=34934
2026-06-11T07:17:48.423896Z  INFO bore_cli::client: listening at bore.pub:34934
2026-06-11T07:18:59.913081Z  INFO proxy{id=ecb2a379-9c8f-4924-9dd7-7eafed90482f}: bore_cli::client: new connection
2026-06-11T07:18:59.978451Z  INFO proxy{id=efca5535-5111-4d2c-a947-232d47f7b732}: bore_cli::client: new connection
2026-06-11T07:19:00.037231Z  INFO proxy{id=ac7ea016-0034-4e6a-aa6a-d82d2bfaea55}: bore_cli::client: new connection
2026-06-11T07:19:00.569697Z  INFO proxy{id=97baeb7b-56d0-4cee-9940-0f6ebeee1507}: bore_cli::client: new connection
2026-06-11T07:19:02.797065Z  INFO proxy{id=750e297d-5f17-470e-98f5-2830911101f2}: bore_cli::client: new connection
2026-06-11T07:19:02.866824Z  INFO proxy{id=11506f4c-bfd4-436d-8461-907f3b10c20b}: bore_cli::client: new connection
2026-06-11T07:19:02.936889Z  INFO proxy{id=11506f4c-bfd4-436d-8461-907f3b10c20b}: bo